# HYZ 2026 Public Colab

Private `HYZ_2026` ana reposunu submodule'lar ile klonlar, Task 1 `.pt` ağırlığını doğrular ve Task 2'nin gerçek yarışma akışını çalıştırır. Task 1 ve Task 3 bu sürümde bilinçli olarak boş payload üretir.


## 1. Colab Secrets

Şunları ekleyin:

- `GITHUB_TOKEN`
- `TEAM_NAME`
- `PASSWORD`
- `EVALUATION_SERVER_URL`

Task 1 `.pt` dosyası Secret değildir: ikinci adımda bilgisayarından seçip Colab'a yüklenecek. Secret değerleri ekrana basılmaz; `GITHUB_TOKEN` yalnızca private repo ve submodule clone işlemi için kullanılır.


In [ ]:
import os
import stat
import subprocess
from pathlib import Path
from google.colab import userdata

REPOSITORY_URL = "https://github.com/RaidynTeam/HYZ_2026.git"
REPOSITORY_BRANCH = "main"
REPOSITORY_DIR = Path("/content/HYZ_2026")
REQUIRED_TASK2_COMMIT = "f746a437b2548b0e99f619105d6cc6f263edc982"

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secrets içinde GITHUB_TOKEN eksik")

os.chdir("/content")
askpass = Path("/content/.hyz_git_askpass.sh")
askpass.write_text(
    '#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) printf %s "$GITHUB_TOKEN" ;; esac\n',
    encoding="utf-8",
)
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
clone_environment = os.environ.copy()
clone_environment.update({
    "GITHUB_TOKEN": github_token,
    "GIT_ASKPASS": str(askpass),
    "GIT_TERMINAL_PROMPT": "0",
})
try:
    if (REPOSITORY_DIR / ".git").is_dir():
        print("Mevcut runtime korunuyor; snapshot ve indirilen kareler silinmeyecek.")
        subprocess.run(
            ["git", "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
            check=True, cwd=REPOSITORY_DIR, env=clone_environment,
        )
    else:
        subprocess.run(
            [
                "git", "clone", "--recurse-submodules",
                "--branch", REPOSITORY_BRANCH, "--single-branch",
                REPOSITORY_URL, str(REPOSITORY_DIR),
            ],
            check=True,
            env=clone_environment,
        )
    subprocess.run(
        ["git", "submodule", "update", "--init", "--recursive"],
        check=True,
        cwd=REPOSITORY_DIR,
        env=clone_environment,
    )
finally:
    askpass.unlink(missing_ok=True)
    clone_environment.pop("GITHUB_TOKEN", None)
    del github_token

os.chdir(REPOSITORY_DIR)
commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], text=True
).strip()
task2_directory = REPOSITORY_DIR / "modules" / "task2_trajectory"
task2_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=task2_directory, text=True
).strip()
task2_contains_guard = subprocess.run(
    ["git", "merge-base", "--is-ancestor", REQUIRED_TASK2_COMMIT, task2_commit],
    cwd=task2_directory,
).returncode == 0
if not task2_contains_guard:
    raise RuntimeError(
        f"Task 2 eski sürüm: {task2_commit[:8]}; gerekli düzeltme: {REQUIRED_TASK2_COMMIT[:8]}"
    )
submodules = subprocess.check_output(
    ["git", "submodule", "status"], text=True
).strip()
print(f"Repository hazır: branch={REPOSITORY_BRANCH} commit={commit}")
print(f"Task 2 causal degraded fallback hazır: {task2_commit[:8]}")
print(submodules)


## 2. Task 1 `.pt` ağırlığını bilgisayardan yükle

Dosya seçici açıldığında bilgisayarından **tek bir** `.pt` dosyası seç. Ağırlık Git'e girmez; Colab oturumunda `/content/models/task1.pt` olarak tutulur. Şimdilik sadece yüklenebilirliği doğrulanır, Task 1 payload'ı yine boş kalır.


In [ ]:
from google.colab import files

TASK1_MODEL_PATH = Path("/content/models/task1.pt")
uploaded = files.upload()
pt_files = [name for name in uploaded if Path(name).suffix.lower() == ".pt"]
if len(uploaded) != 1 or len(pt_files) != 1:
    raise RuntimeError("Bilgisayarından tam olarak bir adet .pt model dosyası seçmelisin")

TASK1_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
temporary_model_path = TASK1_MODEL_PATH.with_suffix(".pt.part")
model_name = pt_files[0]
model_bytes = uploaded[model_name]
if not model_bytes:
    temporary_model_path.unlink(missing_ok=True)
    raise RuntimeError("Task 1 model dosyası boş yüklendi")
temporary_model_path.write_bytes(model_bytes)
temporary_model_path.replace(TASK1_MODEL_PATH)
os.environ["TASK1_MODEL_PATH"] = str(TASK1_MODEL_PATH)
del uploaded, model_bytes
print(f"Task 1 ağırlığı hazır: {TASK1_MODEL_PATH.name} ({TASK1_MODEL_PATH.stat().st_size / 1024 / 1024:.1f} MiB)")


## 3. Python 3.12 ve bağımlılıklar

Ana repo ve submodule bağımlılıkları kilitli dosyadan kurulur.


In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "uv"],
    check=True,
)
subprocess.run(["uv", "python", "install", "3.12"], check=True)
subprocess.run(
    ["uv", "sync", "--python", "3.12", "--frozen"],
    cwd=REPOSITORY_DIR,
    check=True,
)
print("Python 3.12 ortamı ve HYZ paketleri hazır.")


## 4. Task 1 model durumu

Model Colab oturumunda hazır tutulur. Task 1'in yarışma algoritması henüz etkin olmadığı için model yüklenmez/inference çalıştırılmaz ve boş Task 1 payload'ı korunur.


In [ ]:
if not TASK1_MODEL_PATH.is_file():
    raise FileNotFoundError(f"Task 1 model missing: {TASK1_MODEL_PATH}")
print(f"Task 1 .pt dosyası hazır: {TASK1_MODEL_PATH.name}")
print("Task 1/Task 3 boş payload; yalnızca Task 2 gerçek trajectory üretecek.")


## 5. Yarışma ayarları

Secret'lar ignored `config/.env` dosyasına yazılır.


In [ ]:
def required_secret(name):
    value = userdata.get(name)
    if not value:
        raise RuntimeError(f"Colab Secrets içinde {name} eksik")
    return str(value)

def quote_env(value):
    clean = value.replace("\n", "").replace("\r", "")
    return '"' + clean.replace("\\", "\\\\").replace('\"', '\\"') + '"'

environment_values = {
    "TEAM_NAME": required_secret("TEAM_NAME"),
    "PASSWORD": required_secret("PASSWORD"),
    "EVALUATION_SERVER_URL": required_secret("EVALUATION_SERVER_URL").rstrip("/") + "/",
    "SESSION_NAME": "",
}
environment_path = REPOSITORY_DIR / "config" / ".env"
environment_path.write_text(
    "".join(f"{key}={quote_env(value)}\n" for key, value in environment_values.items()),
    encoding="utf-8",
)
environment_path.chmod(0o600)
del environment_values
print("Private config/.env hazır; secret değerleri gösterilmedi. Task 1 ve Task 3 boş payload modunda.")


## 6. Yarışmayı başlat

`START_COMPETITION = True` olunca gerçek server'a gönderim başlar.


In [ ]:
START_COMPETITION = False

if START_COMPETITION:
    model_path = Path("/content/models/task1.pt")
    if not model_path.is_file():
        raise RuntimeError("Task 1 .pt modeli bulunamadı; başlatma iptal edildi")
    print("Gönderim modu: Task 2 gerçek trajectory | Task 1 boş | Task 3 boş")
    process = subprocess.Popen(
        ["uv", "run", "--frozen", "python", "-u", "main.py"],
        cwd=REPOSITORY_DIR,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    exit_code = process.wait()
    if exit_code:
        raise RuntimeError(f"main.py exit code {exit_code}; gerçek hata yukarıda yazıyor")
else:
    print("Güvenli bekleme: START_COMPETITION=False, server'a prediction gönderilmedi.")
